# Hierarchical & Density-Based Clustering

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/clustering/02-hierarchical-and-dbscan

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — clustering beyond spheres

K-means needs `K` up front and only finds blob-shaped clusters. Two alternatives lift those limits.
**DBSCAN** groups points by **density**: a *core* point has many neighbors within radius `eps`,
clusters grow by chaining core points together, and low-density points are labeled **noise**. It finds
**arbitrary shapes**, discovers the number of clusters automatically, and handles outliers — no `K`
needed. **Hierarchical clustering** instead builds a whole tree (**dendrogram**) by repeatedly merging
the closest clusters; you cut the tree at any height to get however many clusters you want, and the
**linkage** rule (single/complete/average) controls their shape. We build both and validate with
`sklearn`/`scipy`.

## DBSCAN: Finding Arbitrary Shapes

Core points have `min_samples` neighbors within `eps`.
Border points are near core points.
Noise points are neither.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, eps in zip(axes, [0.1, 0.2, 0.5]):
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_moons)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    unique_labels = set(labels)
    colors = ['#818cf8', '#14b8a6', '#eab308', '#f43f5e']
    for label in unique_labels:
        mask = labels == label
        if label == -1:
            ax.scatter(X_moons[mask, 0], X_moons[mask, 1], c='#f43f5e', s=20, marker='x', alpha=0.7, label='Noise')
        else:
            ax.scatter(X_moons[mask, 0], X_moons[mask, 1], c=colors[label % len(colors)], s=15, alpha=0.7)
    ax.set_title(f'eps={eps}\n{n_clusters} clusters, {n_noise} noise pts', color='white', fontsize=11)
    ax.set_xlim(-2, 3)
    ax.set_ylim(-1.5, 2)
    ax.set_aspect('equal')
plt.suptitle('DBSCAN: eps Controls Cluster Discovery', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**What to notice:** with a well-chosen `eps` (~0.2) DBSCAN cleanly separates the **two crescent
moons** — the exact shape k-means mangled last lesson. Too **small** an `eps` labels everything as
noise (no point has enough neighbors); too **large** merges the moons into one blob. `eps` is
DBSCAN's key knob, trading noise against merging.

## The library way — DBSCAN succeeds where k-means failed

Concrete payoff: on the same two-moons data where k-means scored ARI ≈ 0.23, DBSCAN recovers the true
clusters. The cell measures it.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score

Xm, ym = make_moons(n_samples=300, noise=0.08, random_state=42)
db = DBSCAN(eps=0.2, min_samples=5).fit_predict(Xm)
ari = adjusted_rand_score(ym, db)
print(f'DBSCAN ARI on two moons: {ari:.3f}   (k-means managed only ~0.23 on this shape)')
assert ari > 0.9, "DBSCAN should recover the crescents"
print('DBSCAN follows arbitrary shapes where k-means cannot ✓')

**What to notice:** DBSCAN's ARI is ~1.0 versus k-means's ~0.23 on the *same* data — because it
grows clusters along **connected density** rather than carving convex regions. When your clusters
aren't blobs, the algorithm's assumptions matter more than any parameter tuning.

In [ ]:
# DBSCAN by hand: core / border / noise on the 9-point example
# (matches the lesson). KEY: min_samples counts the point ITSELF.
import numpy as np
from sklearn.cluster import DBSCAN

pts = np.array([[1,1],[1,2],[2,1],[2,2],[8,8],[8,9],[9,8],[9,9],[5,5]])
eps, min_samples = 2.0, 3

D = np.linalg.norm(pts[:, None] - pts[None], axis=2)   # pairwise distances
neigh_counts = (D <= eps).sum(axis=1)                  # includes self (D=0 <= eps)
is_core = neigh_counts >= min_samples

print(f'eps={eps}, min_samples={min_samples}  (a core point needs {min_samples-1} OTHER neighbors)\n')
for i, p in enumerate(pts):
    others = [tuple(pts[j]) for j in range(len(pts)) if j != i and D[i, j] <= eps]
    kind = 'CORE' if is_core[i] else ('noise' if not others else 'border')
    print(f'  {tuple(p)}: |N_eps| incl. self = {neigh_counts[i]}  neighbors={others}  -> {kind}')

# Cross-check against scikit-learn
labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(pts)
print(f'\nsklearn labels: {labels}   (-1 = noise)')
print('(5,5) is noise:', labels[-1] == -1)


**What to notice:** the by-hand pass labels each point **core** (≥ `min_pts` neighbors within
`eps`), **border** (near a core but not itself dense), or **noise** (neither). Clusters are then the
connected components of core points plus their borders — a purely local, density-driven definition
that needs no global `K`.

## Hierarchical Clustering

Build a dendrogram by merging the closest clusters step by step.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

np.random.seed(42)
n = 50
X_small = np.vstack([np.random.randn(20, 2) + [0, 0],
                     np.random.randn(15, 2) + [4, 0],
                     np.random.randn(15, 2) + [2, 4]])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_small[:, 0], X_small[:, 1], c='#818cf8', s=15, alpha=0.7)
axes[0].set_title('Data Points', color='white')
axes[0].set_aspect('equal')

Z = linkage(X_small, method='ward')
dendrogram(Z, ax=axes[1], color_threshold=7, leaf_font_size=8)
axes[1].set_title('Ward Linkage Dendrogram', color='white')
axes[1].axhline(y=7, color='#f43f5e', linestyle='--', alpha=0.7, label='Cut → 3 clusters')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

**What to notice:** hierarchical clustering builds a **dendrogram** — a tree of merges where the
height of each join is the distance at which two clusters combined. You don't pick `K` in advance;
instead you **cut the tree** at a chosen height, and everything below a cut becomes one cluster.

## Linkage criteria

Agglomerative clustering merges the closest clusters; the **linkage** defines 'closest' (single = nearest pair, complete = farthest pair, ward = min variance increase).

In [ ]:
# Hierarchical by hand: A,B,C,D — distance matrix + per-linkage final-merge height
import numpy as np
from scipy.cluster.hierarchy import linkage

P = {'A': (1,1), 'B': (2,2), 'C': (8,8), 'D': (9,9)}
names = list(P); X4 = np.array(list(P.values()))
D = np.linalg.norm(X4[:, None] - X4[None], axis=2)
print('Distance matrix:')
print('     ' + '   '.join(names))
for i, n in enumerate(names):
    print(f'  {n}  ' + '  '.join(f'{D[i,j]:5.2f}' for j in range(4)))

# Merges 1 & 2: A+B and C+D both at 1.41 (smallest off-diagonal entries)
AB, CD = [0, 1], [2, 3]
cross = D[np.ix_(AB, CD)].ravel()    # the 4 A/B-to-C/D distances
print('\nAfter merging A+B and C+D (both at height 1.41), the {AB}-{CD} cross distances:')
print('  ', np.round(cross, 2).tolist())
print(f'  single  (min)  = {cross.min():.2f}')
print(f'  complete(max)  = {cross.max():.2f}')
print(f'  average (mean) = {cross.mean():.2f}')

# Cross-check the final-merge height that scipy reports for each linkage
print('\nscipy linkage final-merge height:')
for m in ['single', 'complete', 'average']:
    Z = linkage(X4, method=m)
    print(f'  {m:9s}: {Z[-1, 2]:.2f}')


**What to notice:** the **linkage** rule defines the distance *between clusters* and reshapes the
result. **Single** linkage (nearest pair) can *chain* through bridges into long straggly clusters;
**complete** linkage (farthest pair) makes tight compact clusters; **average** linkage balances the
two. Same data, different linkage, different clusters.

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=120, centers=3, cluster_std=0.7, random_state=0)
for method in ['single', 'complete', 'average', 'ward']:
    Z = linkage(X, method=method)
    labels = fcluster(Z, t=3, criterion='maxclust')
    print(f'{method:9s}: {len(np.unique(labels))} clusters, sizes {np.bincount(labels)[1:]}')

**What to notice:** `scipy`'s `fcluster` cuts the dendrogram to produce a flat labeling — the same
merge tree can yield 2, 3, or more clusters depending on where you cut. That flexibility (choose the
resolution after seeing the tree) is hierarchical clustering's signature advantage over k-means.

## Gotchas & tradeoffs

- **DBSCAN's `eps`/`min_samples` are finicky**, and it struggles with clusters of **very different
  density** — one global `eps` can't fit both a dense and a sparse cluster.
- **Hierarchical is `O(n²)` memory (and up to `O(n³)` time).** The distance matrix doesn't scale to
  large `n`.
- **Single linkage chains.** A thin bridge of points can merge two clearly separate clusters — complete
  or average linkage is usually safer.
- **DBSCAN's upside:** no `K` needed, finds arbitrary shapes, and flags outliers as noise — but it
  can't cluster data where *all* densities are similar and clusters touch.

In [ ]:
# DBSCAN uses ONE eps -> clusters of very different density confuse it
from sklearn.datasets import make_blobs
from sklearn.cluster import DBSCAN
Xa, _ = make_blobs(n_samples=150, centers=[[0, 0]], cluster_std=0.3, random_state=0)   # dense
Xb, _ = make_blobs(n_samples=150, centers=[[6, 6]], cluster_std=1.6, random_state=0)   # sparse
Xd = np.vstack([Xa, Xb])
db = DBSCAN(eps=0.4, min_samples=5).fit_predict(Xd)
n_clusters = len(set(db)) - (1 if -1 in db else 0)
print(f'one eps=0.4 on mixed-density data -> {n_clusters} cluster(s), {list(db).count(-1)} noise points')
print('-> the eps that fits the dense blob labels the sparse blob as noise (varying-density weakness)')

**What to notice:** a single `eps` tuned for the **dense** cluster marks most of the **sparse**
cluster as noise — DBSCAN can't adapt its radius per region. Varying-density data is its main blind
spot (HDBSCAN was invented to fix exactly this). Every clustering method has a shape/density regime
where it shines and one where it breaks.

## Key takeaways

- **Hierarchical** clustering builds a dendrogram; cut it at any level for K clusters — no K upfront.
- **Linkage** (single/complete/average/ward) changes cluster shape; ward favors compact clusters.
- **DBSCAN** finds arbitrary shapes by density and labels sparse points as **noise** — no K needed.
- Tune DBSCAN's `eps` via the k-distance elbow; it struggles with varying densities.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — DBSCAN core points

DBSCAN's vocabulary starts here: a point is a **core point** if at least `min_pts` points (counting itself) sit within distance `eps` of it. Cores grow clusters; everything not reachable from a core is noise. Implement the test.

In [ ]:
def is_core_point(X, i, eps, min_pts):
    """Is point i a core point? (its eps-neighborhood, including itself, has >= min_pts points)"""
    X = np.asarray(X, dtype=float)

    # TODO(you): distances from point i to every point
    dists = ...

    # TODO(you): count how many are <= eps and compare to min_pts
    return ...

In [ ]:
# Checks — run me
Xd = np.array([[0.0, 0], [0.5, 0], [1.0, 0], [0.5, 0.5], [10.0, 10.0]])

assert is_core_point(Xd, 1, eps=1.0, min_pts=4), "the dense middle point has 4 neighbors within eps (incl. itself)"
assert not is_core_point(Xd, 4, eps=1.0, min_pts=4), "the isolated point is not core"
assert not is_core_point(Xd, 0, eps=1.0, min_pts=5), "raising min_pts demotes border-ish points"

cores = [i for i in range(len(Xd)) if is_core_point(Xd, i, 1.0, 3)]
assert 4 not in cores and len(cores) >= 3, "the far point is never core at eps=1"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def is_core_point(X, i, eps, min_pts):
    X = np.asarray(X, dtype=float)
    dists = np.linalg.norm(X - X[i], axis=1)
    return int(np.sum(dists <= eps)) >= min_pts
```

</details>

### Exercise 2 — Single vs complete linkage

Hierarchical clustering needs a distance *between clusters*. The two classics:

- **single** linkage: distance of the **closest** pair — chains through narrow bridges
- **complete** linkage: distance of the **farthest** pair — prefers compact, round clusters

Implement both in one function. By definition, single can never exceed complete.

In [ ]:
def linkage_distance(A, B, kind="single"):
    """Distance between clusters A and B (arrays of points)."""
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    # TODO(you): all pairwise distances between A's and B's points
    dists = ...

    # TODO(you): min for single linkage, max for complete
    return ...

In [ ]:
# Checks — run me
A = np.array([[0.0, 0.0], [1.0, 0.0]])
B = np.array([[3.0, 0.0], [5.0, 0.0]])

assert abs(linkage_distance(A, B, "single") - 2.0) < 1e-12, "single = closest pair: |1 - 3|"
assert abs(linkage_distance(A, B, "complete") - 5.0) < 1e-12, "complete = farthest pair: |0 - 5|"
assert linkage_distance(A, B, "single") <= linkage_distance(A, B, "complete"), "single never exceeds complete"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def linkage_distance(A, B, kind="single"):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    dists = np.linalg.norm(A[:, None, :] - B[None, :, :], axis=2)
    return float(dists.min()) if kind == "single" else float(dists.max())
```

</details>